# Time-Resolved Decision Predictability — Discrete-Only BunDLe-Net

Identical analysis to `time_resolved_predictability.ipynb` but using the
**discrete-only** BunDLe-Net run (`b_type=discrete`, no HGF supervision).

Session: `JPAS_0023_20230922` | `decision_strict` | `same_partition` | `latent_dim=3`

**Sections:**
- **Part 1** — Config, data loading, classifier
- **Part 2** — Latent-space geometry
- **Part 3** — Time-resolved accuracy by context (block / congruent / stay-switch / prev-reward)
- **Part 4** — Hybrid vs Discrete comparison
- **Part 5** — Summary and interpretation

## Part 1 — Configuration, Data Loading, and Classifier

In [ ]:
from pathlib import Path
import json, warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.decomposition import PCA

# ── Repo root ────────────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'demos':
    REPO_ROOT = REPO_ROOT.parent

# ── Run directories ──────────────────────────────────────────────────────────
DISC_RUN = REPO_ROOT / (
    'results/grid_search_20260519_013618_same_partition_decision_strict_discrete_only'
    '/run_20260519_013620'
)
HYBRID_RUN = REPO_ROOT / (
    'results/grid_search_20260518_221010_same_partition_decision_strict_alpha_050_070_090'
    '/grid_search_20260518_221026'
    '/run_000_data_path=JPAS_0023_20230922_downsample_fs=30_downsample_method=gaussian'
    '_good_neurons_only=False_apply_hold_tra_64b61128'
)

# ── Pre-computed analysis output directories ─────────────────────────────────
DISC_OUT   = sorted(REPO_ROOT.glob(
    'results/analysis/time_resolved_predictability_discrete_only_*'))[-1]
HYBRID_OUT = REPO_ROOT / 'results/analysis/time_resolved_predictability_20260521_012643'

print(f'Discrete run:    {DISC_RUN}')
print(f'Discrete output: {DISC_OUT}')
print(f'Hybrid output:   {HYBRID_OUT}')

# ── Colour palettes ──────────────────────────────────────────────────────────
LAT_COLORS   = {0:'#2196F3', 1:'#FF5722'}
CONG_PALETTE = {'congruent':'#4CAF50', 'outlier':'#F44336'}
SS_PALETTE   = {'stay':'#1565C0', 'switch':'#E65100'}
BLOCK_PALETTE= {'better left':'#2196F3', 'better right':'#FF5722'}
REW_PALETTE  = {True:'#43A047', False:'#E53935'}
b_labels     = ['choosing left', 'choosing right']

In [ ]:
# ── Load arrays ─────────────────────────────────────────────────────────────
D = DISC_RUN / 'data'
lat_train = np.load(D / 'latent_trajectories_train.npy')
b_train   = np.load(D / 'behaviour_labels_train.npy').astype(int)
lat_val   = np.load(D / 'latent_trajectories_validation.npy')
b_val     = np.load(D / 'behaviour_labels_validation.npy').astype(int)
tid_val   = np.load(D / 'trial_ids_validation.npy')

print(f'lat_train:  {lat_train.shape}  {lat_train.dtype}')
print(f'lat_val:    {lat_val.shape}  {lat_val.dtype}')
print(f'b_train:    {b_train.shape}  unique={np.unique(b_train)}')
print(f'b_val:      {b_val.shape}  unique={np.unique(b_val)}')
print(f'tid_val:    {tid_val.shape}')

In [ ]:
# ── Fit LogisticRegression decoder ──────────────────────────────────────────
clf = LogisticRegression(max_iter=500, C=1.0, random_state=42)
clf.fit(lat_train, b_train)

val_probs   = clf.predict_proba(lat_val)
val_preds   = clf.predict(lat_val)
val_correct = (val_preds == b_val).astype(int)
val_acc     = accuracy_score(b_val, val_preds)
val_bal_acc = balanced_accuracy_score(b_val, val_preds)
chance      = len(b_val[b_val == 1]) / len(b_val)

w     = clf.coef_[0]
b_int = clf.intercept_[0]

print(f'Val accuracy:         {val_acc:.4f}')
print(f'Balanced accuracy:    {val_bal_acc:.4f}')
print(f'Majority baseline:    {chance:.4f}')
print(f'Classifier weights w: {w}')
print(f'Classifier bias b:    {b_int:.4f}')

In [ ]:
# ── Build validation DataFrame ───────────────────────────────────────────────
df_val = pd.DataFrame({
    'y0': lat_val[:,0], 'y1': lat_val[:,1], 'y2': lat_val[:,2],
    'label': b_val.astype(int), 'trial_id': tid_val.astype(int),
    'correct': val_correct, 'pred': val_preds.astype(int),
    'p_class0': val_probs[:,0], 'p_class1': val_probs[:,1],
})
df_val['choice_label']     = df_val['label'].map({0: b_labels[0], 1: b_labels[1]})
df_val['within_trial_idx'] = df_val.groupby('trial_id').cumcount()
nwm = df_val.groupby('trial_id')['within_trial_idx'].transform('max')
df_val['normalized_time']  = df_val['within_trial_idx'] / nwm.clip(lower=1)

# Inject trial metadata from hybrid analysis (same session, same partition, same trial IDs)
h = pd.read_csv(HYBRID_OUT / 'validation_prediction_table_with_metadata_and_margin.csv')
META = ['trial_id','block_label','block_idx','better_side','choice','rewarded',
        'congruent_label','trial_pos_in_block','stay_switch','prev_rewarded','prev_choice']
df_meta = df_val.merge(h[META].drop_duplicates(), on='trial_id', how='left')

# Decision-margin computation
df_meta['signed_dist'] = (lat_val @ w + b_int) / np.linalg.norm(w)
df_meta['class_margin'] = (2*b_val - 1) * df_meta['signed_dist']
df_meta['abs_margin']   = df_meta['class_margin'].abs()

print(f'df_meta: {df_meta.shape}')
print(df_meta[['trial_id','label','correct','normalized_time','congruent_label','stay_switch']].head())

N_BINS = 20
bin_edges = np.linspace(0, 1, N_BINS+1)
bin_ctrs  = (bin_edges[:-1] + bin_edges[1:]) / 2

## Part 2 — Latent-Space Geometry

**Key finding:** The discrete-only latent space is essentially **1-dimensional**.  
PCA explains 99.9999% of variance in the first component (`y₀`), compared to 86.6% for the hybrid run.  
HGF supervision appears to spread the representation across multiple dimensions.

In [ ]:
# PCA geometry
pca = PCA(n_components=3).fit(lat_train)
ev  = pca.explained_variance_ratio_
print(f'PCA explained variance: PC1={ev[0]:.6f}  PC2={ev[1]:.6f}  PC3={ev[2]:.6f}')
print(f'  -> {ev[0]*100:.4f}% of variance in PC1 alone')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
DIM_NAMES = ['y₀', 'y₁', 'y₂']
fig.suptitle(
    f'Discrete-Only — PCA latent geometry\n'
    f'EV: y₀={ev[0]:.4f}  y₁={ev[1]:.6f}  y₂={ev[2]:.6f}',
    fontsize=11, fontweight='bold'
)
y_all  = lat_val
lab_all = b_val
for ax, (xi, yi) in zip(axes, [(0,1),(0,2),(1,2)]):
    for lab in [0, 1]:
        m = lab_all == lab
        ax.scatter(y_all[m,xi], y_all[m,yi], c=LAT_COLORS[lab],
                   alpha=0.12, s=2, rasterized=True,
                   label=b_labels[lab].replace('choosing ',''))
    ax.set_xlabel(DIM_NAMES[xi]); ax.set_ylabel(DIM_NAMES[yi]); ax.legend(fontsize=7, markerscale=4)
fig.tight_layout()
plt.savefig(DISC_OUT/'latent_space_pca_geometry.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3D scatter + decision boundary
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(18, 5))
fig.suptitle(
    'Discrete-Only — Latent-space geometry & logistic decision boundary\n'
    '(grey=boundary; colour=true label)', fontsize=12, y=1.01
)
ax3d = fig.add_subplot(1,4,1, projection='3d')
for lab in [0,1]:
    m = lab_all==lab
    ax3d.scatter(y_all[m,0], y_all[m,1], y_all[m,2], c=LAT_COLORS[lab],
                 alpha=0.12, s=2, label=b_labels[lab].replace('choosing ',''))
g0 = np.linspace(y_all[:,0].min(), y_all[:,0].max(), 24)
g1 = np.linspace(y_all[:,1].min(), y_all[:,1].max(), 24)
G0, G1 = np.meshgrid(g0, g1)
if abs(w[2]) > 1e-8:
    G2 = -(b_int + w[0]*G0 + w[1]*G1) / w[2]
    G2 = np.clip(G2, y_all[:,2].min(), y_all[:,2].max())
    ax3d.plot_surface(G0, G1, G2, alpha=0.28, color='#9E9E9E')
ax3d.set_xlabel('y₀',fontsize=8,labelpad=1); ax3d.set_ylabel('y₁',fontsize=8,labelpad=1)
ax3d.set_zlabel('y₂',fontsize=8,labelpad=1); ax3d.tick_params(labelsize=6)
ax3d.set_title('3D',fontsize=9); ax3d.legend(markerscale=4,fontsize=7)

PROJ = [(0,1,2),(0,2,1),(1,2,0)]
for pi, (xi,yi,fi) in enumerate(PROJ):
    ax = fig.add_subplot(1,4,pi+2)
    for lab in [0,1]:
        m = lab_all==lab
        ax.scatter(y_all[m,xi], y_all[m,yi], c=LAT_COLORS[lab], alpha=0.09, s=1.5, rasterized=True)
    mf = float(y_all[:,fi].mean())
    xlr = np.linspace(y_all[:,xi].min(), y_all[:,xi].max(), 300)
    if abs(w[yi]) > 1e-8:
        ylr = -(w[xi]*xlr + w[fi]*mf + b_int) / w[yi]
        ax.plot(xlr, ylr, color='#424242', lw=2, label=f'boundary ({DIM_NAMES[fi]}=μ)')
    ax.set_xlim(y_all[:,xi].min(), y_all[:,xi].max())
    ax.set_ylim(y_all[:,yi].min(), y_all[:,yi].max())
    ax.set_xlabel(DIM_NAMES[xi],fontsize=9); ax.set_ylabel(DIM_NAMES[yi],fontsize=9)
    ax.set_title(f'{DIM_NAMES[xi]} vs {DIM_NAMES[yi]}\n({DIM_NAMES[fi]}=μ)',fontsize=9)
    ax.legend(
        handles=[Patch(facecolor=LAT_COLORS[l], label=b_labels[l].replace('choosing ','')) for l in [0,1]]
                + [plt.Line2D([0],[0],color='#424242',lw=2,label='boundary')],
        fontsize=7
    )
plt.tight_layout()
plt.savefig(DISC_OUT/'latent_space_decision_boundary.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 3 — Time-Resolved Accuracy by Context

All panels use **normalised decision-strict trial time** on the x-axis (0=trial start, 1=choice point).

- **3a** — Overall time-resolved accuracy (left vs right)
- **3b** — By block context (better left / better right)
- **3c** — Congruent vs Outlier trials
- **3d** — Stay vs Switch choices
- **3e** — Previous reward (rewarded / unrewarded)
- **3f** — Previous × current choice accuracy matrix

In [ ]:
# 3a: overall time-resolved accuracy
bin_idx = pd.cut(df_meta['normalized_time'], bins=bin_edges, labels=False, include_lowest=True)
bin_acc = np.array([df_meta['correct'][bin_idx==i].mean() if (bin_idx==i).any() else np.nan for i in range(N_BINS)])
bin_sem = np.array([stats.sem(df_meta['correct'][bin_idx==i]) if (bin_idx==i).any() else np.nan for i in range(N_BINS)])

fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(bin_ctrs, bin_acc - bin_sem, bin_acc + bin_sem, alpha=0.2, color='steelblue')
ax.plot(bin_ctrs, bin_acc, '-o', color='steelblue', ms=4, lw=2)
ax.axhline(chance, color='grey', ls='--', lw=1.5, label=f'Majority baseline ({chance:.3f})')
ax.axhline(0.5, color='lightgray', ls=':', lw=1)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Normalised decision-strict time'); ax.set_ylabel('Accuracy')
ax.set_title(f'Discrete-Only — Time-resolved accuracy\nval_acc={val_acc:.4f}  bal_acc={val_bal_acc:.4f}')
ax.legend()
fig.tight_layout()
plt.savefig(DISC_OUT/'accuracy_over_decision_strict_time.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3b: by block context
block_types = ['better left', 'better right']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Discrete-Only — C1: Accuracy by block context', fontsize=14)
for bt in block_types:
    sub = df_meta[df_meta['better_side'] == bt]
    n_tr = sub['trial_id'].nunique()
    ba, bs = [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        m = (sub['normalized_time'] >= lo) & (sub['normalized_time'] < hi)
        v = sub.loc[m, 'correct']
        ba.append(v.mean() if len(v) >= 3 else np.nan)
        bs.append(v.std()/np.sqrt(len(v)) if len(v) >= 3 else np.nan)
    ba = np.array(ba); bs = np.array(bs)
    axes[0].plot(bin_ctrs, ba, '-o', color=BLOCK_PALETTE[bt],
                 label=f'{bt} (n={n_tr})', lw=2, ms=4)
    axes[0].fill_between(bin_ctrs, ba-bs, ba+bs, color=BLOCK_PALETTE[bt], alpha=0.2)
axes[0].axhline(chance, color='grey', ls=':'); axes[0].set_ylim(0, 1)
axes[0].set_xlabel('Normalised time'); axes[0].set_ylabel('Accuracy'); axes[0].legend()
means_b = [df_meta[df_meta['better_side']==bt]['correct'].mean() for bt in block_types]
cis_b   = [1.96*df_meta[df_meta['better_side']==bt]['correct'].sem() for bt in block_types]
axes[1].bar(range(len(block_types)), means_b,
            color=[BLOCK_PALETTE[b] for b in block_types], alpha=0.85)
axes[1].errorbar(range(len(block_types)), means_b, yerr=cis_b,
                 fmt='none', color='black', capsize=6)
axes[1].axhline(chance, color='grey', ls=':')
axes[1].set_xticks(range(len(block_types))); axes[1].set_xticklabels(block_types)
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel('Mean accuracy')
plt.tight_layout()
plt.savefig(DISC_OUT/'accuracy_by_block_context.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3c: congruent vs outlier
cong_types = ['congruent', 'outlier']
fig, ax = plt.subplots(figsize=(8, 4))
ax.axhline(chance, color='grey', ls=':', lw=1.5, label=f'Baseline ({chance:.3f})')
for ct in cong_types:
    sub = df_meta[df_meta['congruent_label'] == ct]
    n_tr = sub['trial_id'].nunique()
    ba, bs = [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        m = (sub['normalized_time'] >= lo) & (sub['normalized_time'] < hi)
        v = sub.loc[m, 'correct']
        ba.append(v.mean() if len(v) >= 3 else np.nan)
        bs.append(v.std()/np.sqrt(len(v)) if len(v) >= 3 else np.nan)
    ba = np.array(ba); bs = np.array(bs)
    ax.plot(bin_ctrs, ba, '-o', color=CONG_PALETTE[ct],
            label=f'{ct} (n={n_tr})', lw=2, ms=4)
    ax.fill_between(bin_ctrs, ba-bs, ba+bs, color=CONG_PALETTE[ct], alpha=0.2)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Normalised time'); ax.set_ylabel('Accuracy')
ax.set_title('Discrete-Only — Congruent vs Outlier accuracy')
ax.legend(); fig.tight_layout()
plt.savefig(DISC_OUT/'accuracy_congruent_vs_outlier.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3d: stay vs switch
fig, ax = plt.subplots(figsize=(8, 4))
ax.axhline(chance, color='grey', ls=':', lw=1.5, label=f'Baseline ({chance:.3f})')
for ss in ['stay', 'switch']:
    sub = df_meta[df_meta['stay_switch'] == ss]
    n_tr = sub['trial_id'].nunique()
    ba, bs = [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        m = (sub['normalized_time'] >= lo) & (sub['normalized_time'] < hi)
        v = sub.loc[m, 'correct']
        ba.append(v.mean() if len(v) >= 3 else np.nan)
        bs.append(v.std()/np.sqrt(len(v)) if len(v) >= 3 else np.nan)
    ba = np.array(ba); bs = np.array(bs)
    ax.plot(bin_ctrs, ba, '-o', color=SS_PALETTE[ss],
            label=f'{ss} (n={n_tr})', lw=2, ms=4)
    ax.fill_between(bin_ctrs, ba-bs, ba+bs, color=SS_PALETTE[ss], alpha=0.2)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Normalised time'); ax.set_ylabel('Accuracy')
ax.set_title('Discrete-Only — Stay vs Switch accuracy')
ax.legend(); fig.tight_layout()
plt.savefig(DISC_OUT/'accuracy_over_time_stay_vs_switch.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3e: previous reward
fig, ax = plt.subplots(figsize=(8, 4))
ax.axhline(chance, color='grey', ls=':', lw=1.5, label=f'Baseline ({chance:.3f})')
for rw in [True, False]:
    sub = df_meta[df_meta['prev_rewarded'] == rw]
    n_tr = sub['trial_id'].nunique()
    ba, bs = [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        m = (sub['normalized_time'] >= lo) & (sub['normalized_time'] < hi)
        v = sub.loc[m, 'correct']
        ba.append(v.mean() if len(v) >= 3 else np.nan)
        bs.append(v.std()/np.sqrt(len(v)) if len(v) >= 3 else np.nan)
    ba = np.array(ba); bs = np.array(bs)
    ax.plot(bin_ctrs, ba, '-o', color=REW_PALETTE[rw],
            label=f'prev rewarded={rw} (n={n_tr})', lw=2, ms=4)
    ax.fill_between(bin_ctrs, ba-bs, ba+bs, color=REW_PALETTE[rw], alpha=0.2)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Normalised time'); ax.set_ylabel('Accuracy')
ax.set_title('Discrete-Only — Accuracy by previous reward')
ax.legend(); fig.tight_layout()
plt.savefig(DISC_OUT/'accuracy_over_time_by_previous_reward.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3f: previous x current choice matrix
choices = ['l', 'r']
mat_acc = np.full((2, 2), np.nan)
mat_ntr = np.zeros((2, 2), dtype=int)
for ri, pc in enumerate(choices):
    for ci, cc in enumerate(choices):
        sub = df_meta[(df_meta['prev_choice'] == pc) & (df_meta['choice'] == cc)]
        if len(sub) > 0:
            mat_acc[ri, ci] = sub['correct'].mean()
            mat_ntr[ri, ci] = sub['trial_id'].nunique()
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(mat_acc, vmin=0.4, vmax=1, cmap='RdYlGn')
for ri in range(2):
    for ci in range(2):
        ax.text(ci, ri, f'{mat_acc[ri,ci]:.3f}\n(n={mat_ntr[ri,ci]}tr)',
                ha='center', va='center', fontsize=10)
ax.set_xticks([0,1]); ax.set_xticklabels(['curr=L','curr=R'])
ax.set_yticks([0,1]); ax.set_yticklabels(['prev=L','prev=R'])
ax.set_title('Discrete-Only — Prev/Current choice accuracy')
plt.colorbar(im, ax=ax, label='Accuracy')
fig.tight_layout()
plt.savefig(DISC_OUT/'accuracy_previous_current_choice_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Margin distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Discrete-Only — Decision boundary margin distributions', fontsize=12)
for ax, (col, cats, pal) in zip(axes, [
    ('congruent_label', ['congruent','outlier'], CONG_PALETTE),
    ('stay_switch',     ['stay','switch'],        SS_PALETTE)
]):
    for cat in cats:
        v = df_meta[df_meta[col] == cat]['class_margin']
        ax.hist(v, bins=50, alpha=0.5, color=pal[cat],
                label=f'{cat} (μ={v.mean():.3f})', density=True)
    ax.axvline(0, color='black', lw=0.8, ls='--')
    ax.set_xlabel('class_margin (signed distance)'); ax.legend(fontsize=8)
fig.tight_layout()
plt.savefig(DISC_OUT/'margin_distributions_by_group.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 4 — Hybrid vs Discrete Comparison

Both models trained on the **same session** (`JPAS_0023_20230922`), same trial partition,
same `latent_dim=3`, same `decision_strict` / `same_partition` config.

- **Hybrid**: supervised with HGF belief (`alpha=0.5`, continuous stream)
- **Discrete-only**: supervised with choice label only (`b_type=discrete`)

Key differences:
| | Hybrid | Discrete-only |
|---|---|---|
| `b_type` | `hybrid` | `discrete` |
| HGF alpha | 0.5 | — |
| Latent geometry | 2D (PC1=86.6%, PC2=13.4%) | ~1D (PC1=99.999%) |
| Val accuracy | 0.8111 | 0.8076 |
| Congruent acc. | 0.8955 | 0.8842 |
| Outlier acc. | 0.5714 | 0.5900 |

> **Interpretation:** HGF supervision does not significantly boost discrete choice  
> decoding accuracy, but expands the latent representation from 1D to 2D — suggesting  
> that continuous belief tracking uses a geometrically distinct dimension than choice identity.

In [ ]:
# Load comparison JSON (pre-computed)
with open(DISC_OUT / 'hybrid_vs_discrete_summary.json') as f:
    comp = json.load(f)

H = comp['hybrid']
Dc = comp['discrete']

# Print comparison table
print(f"{'Metric':35s}  {'Hybrid':>10s}  {'Discrete':>10s}  {'Diff':>8s}")
print("─"*70)
rows = [
    ('Val accuracy',      H['val_acc'],       Dc['val_acc']),
    ('Balanced accuracy', H['bal_acc'],        Dc['bal_acc']),
    ('Majority baseline', H['majority_baseline'], Dc['majority_baseline']),
    ('Congruent acc.',    H['cong_acc'],       Dc['cong_acc']),
    ('Outlier acc.',      H['outlier_acc'],    Dc['outlier_acc']),
    ('Stay acc.',         H['stay_acc'],       Dc['stay_acc']),
    ('Switch acc.',       H['switch_acc'],     Dc['switch_acc']),
    ('Prev-rewarded',     H['prev_rew_acc'],   Dc['prev_rew_acc']),
    ('Prev-unrewarded',   H['prev_norew_acc'], Dc['prev_norew_acc']),
    ('Margin (congruent)',H['margin_cong'],    Dc['margin_cong']),
    ('Margin (outlier)',  H['margin_outlier'], Dc['margin_outlier']),
    ('PCA PC1 variance',  H['pca_ev'][0],      Dc['pca_ev'][0]),
    ('PCA PC2 variance',  H['pca_ev'][1],      Dc['pca_ev'][1]),
]
for lbl, hv, dv in rows:
    print(f"  {lbl:33s}  {hv:10.4f}  {dv:10.4f}  {dv-hv:+8.4f}")

In [ ]:
# Hybrid vs discrete comparison figure
COLORS = {'hybrid':'#7B1FA2', 'discrete':'#0288D1'}
LABELS = {'hybrid':'Hybrid (HGF α=0.5)', 'discrete':'Discrete-only'}

# Load time-resolved bins for both runs
models = {}
for tag, out_dir in [('hybrid', HYBRID_OUT), ('discrete', DISC_OUT)]:
    df_t = pd.read_csv(out_dir / 'validation_prediction_table_with_metadata_and_margin.csv')
    bid  = pd.cut(df_t['normalized_time'], bins=bin_edges, labels=False, include_lowest=True)
    ba   = np.array([df_t['correct'][bid==i].mean() if (bid==i).any() else np.nan for i in range(N_BINS)])
    bs   = np.array([stats.sem(df_t['correct'][bid==i]) if (bid==i).any() else np.nan for i in range(N_BINS)])
    models[tag] = {'bin_acc': ba, 'bin_sem': bs, 'df': df_t}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(
    'Hybrid vs Discrete-Only BunDLe-Net — Same session/split/dim\n'
    'JPAS_0023_20230922  |  decision_strict  |  same_partition  |  latent_dim=3\n'
    '(one run each; not yet replicated across seeds)',
    fontsize=11, fontweight='bold'
)
width = 0.35

# Time-resolved
ax = axes[0,0]
for tag, m in models.items():
    ax.fill_between(bin_ctrs, m['bin_acc']-m['bin_sem'], m['bin_acc']+m['bin_sem'],
                    color=COLORS[tag], alpha=0.18)
    ax.plot(bin_ctrs, m['bin_acc'], '-o', color=COLORS[tag], ms=4, lw=2,
            label=f"{LABELS[tag]} ({comp[tag]['val_acc']:.4f})")
ax.axhline(chance, color='grey', ls='--', alpha=0.7, label=f'Baseline ({chance:.2f})')
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.legend(fontsize=8)
ax.set_xlabel('Normalised time'); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy over time'); ax.grid(alpha=0.3)

# Congruent vs outlier
ax = axes[0,1]
cats = ['congruent', 'outlier']
x = np.arange(len(cats))
for i, (tag, m) in enumerate(models.items()):
    vals = [m['df'][m['df']['congruent_label']==c]['correct'].mean() for c in cats]
    ax.bar(x+(i-0.5)*width, vals, width, color=COLORS[tag], alpha=0.85, label=LABELS[tag])
ax.axhline(chance, color='grey', ls='--', alpha=0.6)
ax.set_xticks(x); ax.set_xticklabels(cats); ax.set_ylim(0, 1.05)
ax.set_ylabel('Mean accuracy'); ax.set_title('Congruent vs Outlier'); ax.legend(fontsize=8)

# Stay vs switch
ax = axes[0,2]
cats2 = ['stay', 'switch']
x = np.arange(len(cats2))
for i, (tag, m) in enumerate(models.items()):
    vals = [m['df'][m['df']['stay_switch']==c]['correct'].mean() for c in cats2]
    ax.bar(x+(i-0.5)*width, vals, width, color=COLORS[tag], alpha=0.85, label=LABELS[tag])
ax.axhline(chance, color='grey', ls='--', alpha=0.6)
ax.set_xticks(x); ax.set_xticklabels(cats2); ax.set_ylim(0, 1.05)
ax.set_ylabel('Mean accuracy'); ax.set_title('Stay vs Switch'); ax.legend(fontsize=8)

# Previous reward
ax = axes[1,0]
cats3 = ['prev rewarded', 'prev unrewarded']
x = np.arange(2)
for i, (tag, m) in enumerate(models.items()):
    vals = [m['df'][m['df']['prev_rewarded']==r]['correct'].mean() for r in [True, False]]
    ax.bar(x+(i-0.5)*width, vals, width, color=COLORS[tag], alpha=0.85, label=LABELS[tag])
ax.axhline(chance, color='grey', ls='--', alpha=0.6)
ax.set_xticks(x); ax.set_xticklabels(cats3); ax.set_ylim(0, 1.05)
ax.set_ylabel('Mean accuracy'); ax.set_title('Previous Reward'); ax.legend(fontsize=8)

# Margin
ax = axes[1,1]
cats4 = ['cong', 'outlier', 'stay', 'switch']
h_mg = [H['margin_cong'], H['margin_outlier'], H['margin_stay'], H['margin_switch']]
d_mg = [Dc['margin_cong'], Dc['margin_outlier'], Dc['margin_stay'], Dc['margin_switch']]
x4 = np.arange(len(cats4))
ax.bar(x4-width/2, h_mg, width, color=COLORS['hybrid'],  alpha=0.85, label=LABELS['hybrid'])
ax.bar(x4+width/2, d_mg, width, color=COLORS['discrete'],alpha=0.85, label=LABELS['discrete'])
ax.set_xticks(x4); ax.set_xticklabels(cats4); ax.set_ylabel('Mean |margin|')
ax.set_title('Decision margin by group'); ax.legend(fontsize=8)

# PCA
ax = axes[1,2]
x5 = np.arange(3)
ax.bar(x5-width/2, H['pca_ev'],  width, color=COLORS['hybrid'],  alpha=0.85, label=LABELS['hybrid'])
ax.bar(x5+width/2, Dc['pca_ev'], width, color=COLORS['discrete'],alpha=0.85, label=LABELS['discrete'])
ax.set_xticks(x5); ax.set_xticklabels(['PC1','PC2','PC3'])
ax.set_ylabel('Explained variance'); ax.set_title('PCA latent variance'); ax.legend(fontsize=8)
ax.set_yscale('log')

plt.tight_layout()
plt.savefig(DISC_OUT/'hybrid_vs_discrete_context_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 5 — Interpretation and Summary

### Key findings (discrete-only run)

1. **Val accuracy: 0.8076** — Almost identical to the hybrid model (0.8111), despite using no HGF belief  
   supervision. Choice identity alone is sufficient to achieve high decoding accuracy from the latent space.

2. **Context sensitivity: congruent=0.884, outlier=0.590** — The same pattern seen in the hybrid model.  
   Congruent trials (choice matches the better side) are decoded accurately; outlier trials  
   (choice contradicts the better side, e.g., choosing the worse arm) are near-baseline.  
   This implies the neural representation is primarily organised around **which side the animal  
   chose**, not whether the choice was optimal — so outlier predictions are "correct" according to  
   the latent state even though they look wrong from a reward perspective.

3. **Stay=0.862, switch=0.575** — Switches are near-chance. Consistent with point 2: the latent  
   captures the current choice direction, not the transition.

4. **Latent geometry: near 1D** — PCA PC1 explains 99.9999% of variance in the discrete-only  
   latent, vs 86.6% in the hybrid. HGF supervision expands the representation into a 2nd  
   orthogonal dimension — likely encoding belief/confidence, not choice direction.

5. **Margin: hybrid > discrete** — The hybrid model places outlier/switch trials further from  
   the boundary (|margin|_cong = 0.573 vs 0.464). HGF belief may pull "confident" trials  
   further from the boundary.

### Caveats

- Results from **one model run each** (no seed replication). Noise could explain  
  small differences (±0.01 accuracy).  
- The analysis reuses trial metadata from the hybrid CSV (same session, same partition —  
  verified identical `trial_id` and `label` arrays).
- Generalisation across sessions not yet assessed.

In [ ]:
# Print summary JSON
with open(DISC_OUT / 'summary.json') as f:
    summary = json.load(f)

print('Discrete-only summary:')
for k, v in summary.items():
    if k not in ('bootstrap_rarity_control', 'run', 'coef_w'):
        print(f'  {k:28s}: {v}')
print()
print('Figures saved to:', DISC_OUT)

## Part K — Decision-Strict Interval Boundary Verification

Confirms the exact temporal meaning of the `normalized_time` axis used in all figures above.

**Method:** Reload the raw dataset to recover `behavioral_time` (ms timestamps per neuronal frame),
then match `ds.trial_start_indices` against `trial['start']` vs `t_chosen[prev]+1` from
`metrics.json`.

**Key findings:**
- `normalized_time = 0.0` → `trial['start']` (animal re-enters trial zone, after prior reward collection)
- `normalized_time ≈ 0.71` (median) → `t_chosen` (actual choice action, **not** the end of the axis)
- `normalized_time = 1.0` → ~991 ms after `t_chosen` (reward delivery period, not a choice boundary)

> **Label contamination note:** `decision_strict` labels frames `[t_chosen[prev]+1 … t_chosen[N]]`
> as trial N's choice. Because trial *segments* extend ~1 s past `t_chosen[N]`, the frames in
> `(t_chosen[N], segment_end]` carry the **next** trial's choice label. This affects ~17% of
> validation trials (label change at median normalized_time ≈ 0.55).

In [ ]:

# ── K: Decision-Strict Interval Boundary Verification ────────────────────────
import sys
sys.path.insert(0, str(REPO_ROOT))
from ncmcm.data_loaders.bandit_task import BanditTaskNeuroPixelsDataset

_ds_path = REPO_ROOT / 'datasets/raw/twoArmBandit/JPAS_0023_20230922'

print("Reloading dataset for behavioral_time verification (cached — fast) …")
ds_ver = BanditTaskNeuroPixelsDataset(
    data_path=str(_ds_path),
    downsample_fs=30, downsample_method='gaussian',
    good_neurons_only=False, b_mode='decision_strict',
)
print(f"  frames: {ds_ver.x.shape[1]}  trial_start_indices: {len(ds_ver.trial_start_indices)}")

# ── Load usable trials from metrics.json ─────────────────────────────────────
with open(_ds_path / 'metrics.json') as _f:
    _m = json.load(_f)
_usable = sorted(
    [t for t in _m['metrics']['trials']
     if t.get('start') is not None and t.get('t chosen') is not None
     and t.get('choice', '').lower() in ['l', 'r']],
    key=lambda t: t['start']
)
_exp_raw    = np.array([int(t['start'])    for t in _usable], dtype=float)
_exp_chosen = np.array([int(t['t chosen']) for t in _usable], dtype=float)
_exp_strict = np.array(
    [int(_usable[0]['start'])] +
    [int(_usable[j-1]['t chosen']) + 1 for j in range(1, len(_usable))],
    dtype=float
)

# ── Align trial_start_indices to nearest raw trial['start'] ──────────────────
_tsi      = ds_ver.trial_start_indices
_obs_ms   = ds_ver.behavioral_time[_tsi]
_frame_ms = 1000.0 / 30.0
_best_j   = np.argmin(np.abs(_obs_ms[:, None] - _exp_raw[None, :]), axis=1)
_d_raw    = _obs_ms - _exp_raw[_best_j]
_d_strict = _obs_ms - _exp_strict[_best_j]

# ── Post-choice extension per segment ────────────────────────────────────────
_tsi_last      = np.concatenate([_tsi[1:] - 1, [ds_ver.x.shape[1] - 1]])
_obs_end       = ds_ver.behavioral_time[_tsi_last]
_post_ms_vec   = _obs_end - _exp_chosen[_best_j]
_total_seg     = _obs_end - _obs_ms
_t_chosen_frac = np.clip(1 - _post_ms_vec / _total_seg.clip(min=1), 0, 1)

# ── Within-trial label contamination (validation set) ────────────────────────
_changes = []
for _tid in sorted(df_meta['trial_id'].unique()):
    _labs = df_meta.loc[df_meta['trial_id'] == _tid, 'label'].values
    if len(np.unique(_labs)) > 1:
        _ch = np.where(np.diff(_labs) != 0)[0]
        _changes.append((_ch[0] + 1) / len(_labs))
_n_mixed = len(_changes)
_n_val   = df_meta['trial_id'].nunique()
_lc      = np.array(_changes) if _changes else np.array([np.nan])

# ── Print results ─────────────────────────────────────────────────────────────
print(f"\n── Alignment ──")
print(f"  obs_start vs trial['start']:   mean Δ = {_d_raw.mean():+.1f} ms  "
      f"max|Δ| = {np.abs(_d_raw).max():.1f} ms  all≤1frame = {np.all(np.abs(_d_raw) <= _frame_ms)}")
print(f"  obs_start vs t_chosen[prev]+1: mean Δ = {_d_strict.mean():+.1f} ms  (NOT aligned)")
print(f"\n── Temporal structure ──")
print(f"  t_chosen at normalized_time: median = {np.median(_t_chosen_frac):.3f}  "
      f"(mean = {np.mean(_t_chosen_frac):.3f})")
print(f"  post-choice extension: median = {np.median(_post_ms_vec):.0f} ms  "
      f"(min = {_post_ms_vec.min():.0f}  max = {_post_ms_vec.max():.0f} ms)")
print(f"\n── Label contamination (val set) ──")
print(f"  Trials with mid-segment label change: {_n_mixed}/{_n_val} "
      f"({100*_n_mixed/_n_val:.0f}%)")
if not np.isnan(_lc[0]):
    print(f"  Change at normalized_time: median = {np.median(_lc):.3f}  mean = {_lc.mean():.3f}")

# ── Figure ────────────────────────────────────────────────────────────────────
_fig, _ax = plt.subplots(1, 3, figsize=(14, 4))
_fig.suptitle('Part K — Decision-strict boundary verification (discrete-only)', fontweight='bold')

# Panel A: start-alignment histogram
_ax[0].hist(_d_raw, bins=30, color='steelblue', edgecolor='white', lw=0.5)
_ax[0].axvline(0, color='red', lw=1.5, ls='--', label='exact')
_ax[0].axvline( _frame_ms, color='orange', lw=1, ls=':', label=f'±1 frame')
_ax[0].axvline(-_frame_ms, color='orange', lw=1, ls=':')
_ax[0].set_xlabel("obs_start − trial['start']  (ms)")
_ax[0].set_ylabel('# segments')
_ax[0].set_title(f"Start alignment\nmax|Δ| = {np.abs(_d_raw).max():.1f} ms  (frame = {_frame_ms:.1f} ms)")
_ax[0].legend(fontsize=8)

# Panel B: where t_chosen falls in normalized_time
_ax[1].hist(_t_chosen_frac, bins=25, color='#E65100', edgecolor='white', lw=0.5)
_ax[1].axvline(np.median(_t_chosen_frac), color='black', lw=2, ls='--',
               label=f'median = {np.median(_t_chosen_frac):.2f}')
_ax[1].set_xlabel('normalized_time at t_chosen')
_ax[1].set_ylabel('# segments')
_ax[1].set_title('t_chosen position within segment\n(choice ≠ end-of-segment)')
_ax[1].legend(fontsize=9)

# Panel C: time-resolved accuracy with t_chosen annotation
_bidx = pd.cut(df_meta['normalized_time'], bins=bin_edges, labels=False, include_lowest=True)
_ba = np.array([df_meta['correct'][_bidx == i].mean() if (_bidx == i).any() else np.nan
                for i in range(N_BINS)])
_bs = np.array([stats.sem(df_meta['correct'][_bidx == i]) if (_bidx == i).any() else np.nan
                for i in range(N_BINS)])
_ax[2].fill_between(bin_ctrs, _ba - _bs, _ba + _bs, alpha=0.2, color='steelblue')
_ax[2].plot(bin_ctrs, _ba, '-o', color='steelblue', ms=4, lw=2, label='accuracy')
_ax[2].axhline(chance, color='grey', ls='--', lw=1, label='baseline')
_ax[2].axvline(np.median(_t_chosen_frac), color='#E65100', lw=2, ls='--',
               label=f't_chosen (median={np.median(_t_chosen_frac):.2f})')
_ax[2].set_xlim(0, 1); _ax[2].set_ylim(0, 1)
_ax[2].set_xlabel('normalized_time')
_ax[2].set_ylabel('Accuracy')
_ax[2].set_title('Accuracy with t_chosen annotation\n(orange dashed = choice action)')
_ax[2].legend(fontsize=8)

_fig.tight_layout()
plt.savefig(DISC_OUT / 'boundary_verification.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n── Corrected axis label ──")
print(f"  'Normalised within-trial window index'")
print(f"  '(0 = trial entry · ~{np.median(_t_chosen_frac):.1f} = choice action · "
      f"1 = ~{np.median(_post_ms_vec)/1000:.1f} s post-choice)'")
